# 03. Train / Test 분할

채널별로 게시 시간순 정렬 후, 채널마다 앞 80%를 train / 뒤 20%를 test로 분할한다.
모든 채널이 train/test 양쪽에 포함된다 (채널별 기간 외삽 평가).

In [ ]:
import pandas as pd
from pathlib import Path

pd.set_option('display.max_columns', None)

BASE_PATH = Path("../../data")

# 설정
TRAIN_RATIO = 0.8          # 채널별 시간순 train 비율 (8:2)
START_DATE = '2025-01-01'  # 분석 대상 시작일 (이 날짜 이후 영상만 사용)

## 1. 데이터 로드

In [ ]:
df = pd.read_csv(BASE_PATH / "processed/video_processed.csv", sep="\x01", low_memory=False)
df['published_at'] = pd.to_datetime(df['published_at'])

print(f"shape: {df.shape}")
print(f"기간: {df['published_at'].min().date()} ~ {df['published_at'].max().date()}")

shape: (14823, 99)
기간: 2022-01-01 ~ 2025-12-31


In [ ]:
# 분석 대상 기간: START_DATE 이후 영상만 사용
df = df[df['published_at'] >= START_DATE]

print(f"shape: {df.shape}")
print(f"기간: {df['published_at'].min().date()} ~ {df['published_at'].max().date()}")

## 2. Train / Test 분할

In [ ]:
df_sorted = df.sort_values('published_at').reset_index(drop=True)

# 채널별 시간순 분할: 채널마다 앞 TRAIN_RATIO 만큼 train, 나머지 test
train_parts, test_parts = [], []
for ch_id, grp in df_sorted.groupby('channel_id'):
    grp = grp.sort_values('published_at')
    n_train = max(int(len(grp) * TRAIN_RATIO), 1)  # 최소 1개는 train
    train_parts.append(grp.iloc[:n_train])
    test_parts.append(grp.iloc[n_train:])

train = pd.concat(train_parts).reset_index(drop=True)
test = pd.concat(test_parts).reset_index(drop=True)

print(f"채널별 시간순 분할 (train {TRAIN_RATIO:.0%} / test {1-TRAIN_RATIO:.0%})")
print(f"Train: {train.shape}  |  {train['published_at'].min().date()} ~ {train['published_at'].max().date()}")
print(f"Test : {test.shape}  |  {test['published_at'].min().date()} ~ {test['published_at'].max().date()}")

## 3. 검증

In [6]:
tr_ch = set(train['channel_id'].unique())
te_ch = set(test['channel_id'].unique())

print(f"Train 채널: {len(tr_ch)}  |  Test 채널: {len(te_ch)}")
print(f"양쪽 공통  : {len(tr_ch & te_ch)}")
print(f"Train에만  : {len(tr_ch - te_ch)}")
print(f"Test에만   : {len(te_ch - tr_ch)}")

Train 채널: 95  |  Test 채널: 95
양쪽 공통  : 95
Train에만  : 0
Test에만   : 0


## 4. 저장

In [7]:
split_dir = BASE_PATH / "splits"
split_dir.mkdir(exist_ok=True)

train.to_csv(split_dir / "train.csv", sep="\x01", index=False)
test.to_csv(split_dir / "test.csv", sep="\x01", index=False)

# 저장 검증
tr_check = pd.read_csv(split_dir / "train.csv", sep="\x01", nrows=3)
te_check = pd.read_csv(split_dir / "test.csv", sep="\x01", nrows=3)

print(f"저장 완료: {split_dir.resolve()}")
print(f"train.csv: {tr_check.shape[1]} cols  |  test.csv: {te_check.shape[1]} cols")

저장 완료: C:\Users\Dell5371\Desktop\ML프로젝트_refator\data\splits
train.csv: 99 cols  |  test.csv: 99 cols
